In [1]:
# Install LangChain, Google's integration, and the FAISS CPU engine
#!pip install -q -U langchain langchain-community langchain-google-genai faiss-cpu

In [3]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
#from google.colab import userdata
from dotenv import load_dotenv

import os
load_dotenv()
# 1. Secure Authentication
api_key = os.getenv('GEMINI_API_KEY')

# 2. Initialize the Google Embedding Model
embedder = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# 3. Simulate our chunked, metadata-rich documents from Session 1.3
chunks = [
    Document(
        page_content="Tony has proposed moving to a distributed architecture to handle the load of 500 PDFs.",
        metadata={"source": "Extract_Once_Audit.pdf", "chapter": "Engineering", "chunk_id": "CHK-001"}
    ),
    Document(
        page_content="Marten noted a critical issue: 15% of scanned PDF documents are failing the OCR pipeline.",
        metadata={"source": "Extract_Once_Audit.pdf", "chapter": "Compliance", "chunk_id": "CHK-002"}
    ),
    Document(
        page_content="Jaymin is leading the budget review. The estimated cost for Q1 scaling is $45,000.",
        metadata={"source": "Extract_Once_Audit.pdf", "chapter": "Financials", "chunk_id": "CHK-003"}
    ),
    Document(
        page_content="The annual company retreat will be held in Hawaii this year, featuring surfing lessons.",
        metadata={"source": "HR_Memo.pdf", "chapter": "Culture", "chunk_id": "CHK-004"}
    )
]

print("[System] Simulated Document Chunks Loaded. Ready for FAISS Ingestion.")

[System] Simulated Document Chunks Loaded. Ready for FAISS Ingestion.


In [4]:
text_embedding=embedder.embed_query(
    "what is the estimate cost for 01 scaling"
)
print("embedding dimension:",len(text_embedding))

embedding dimension: 3072


In [5]:
document_embediing=embedder.embed_documents(
    [doc.page_content for doc in chunks]
)

print("embedding dimension:",len(document_embediing[0]))
print("number of embeddings:",len(document_embediing))

embedding dimension: 3072
number of embeddings: 4


In [6]:
print("Initializing FAISS Vector Database...\n")

# This single line of code is doing millions of mathematical operations.
# It creates the index and populates it with our embedded documents.
vectorstore = FAISS.from_documents(documents=chunks, embedding=embedder)

print("[Success] FAISS Index built and populated in RAM.")

Initializing FAISS Vector Database...

[Success] FAISS Index built and populated in RAM.


In [7]:
# The user's query
query = "How much money do we need to fix the pipeline?"

print(f"Executing ANN Vector Search for: '{query}'\n")

# We perform the search, asking for the top 2 closest vectors (k=2)
# FAISS embeds the query, calculates the Cosine distances in the graph, and returns the matches.
results = vectorstore.similarity_search(query, k=2)

print("=== SEARCH RESULTS ===")
for i, doc in enumerate(results):
    print(f"\n[Match #{i+1}]")
    print(f"Content: {doc.page_content}")
    print(f"Retrieved Metadata: {doc.metadata}")

print("\n-> Notice how the #1 result is Jaymin's chunk about the $45,000 budget, even though the word 'money' wasn't in the text!")

Executing ANN Vector Search for: 'How much money do we need to fix the pipeline?'

=== SEARCH RESULTS ===

[Match #1]
Content: Jaymin is leading the budget review. The estimated cost for Q1 scaling is $45,000.
Retrieved Metadata: {'source': 'Extract_Once_Audit.pdf', 'chapter': 'Financials', 'chunk_id': 'CHK-003'}

[Match #2]
Content: Marten noted a critical issue: 15% of scanned PDF documents are failing the OCR pipeline.
Retrieved Metadata: {'source': 'Extract_Once_Audit.pdf', 'chapter': 'Compliance', 'chunk_id': 'CHK-002'}

-> Notice how the #1 result is Jaymin's chunk about the $45,000 budget, even though the word 'money' wasn't in the text!


#now the faiss in ram next we need to save it in harddisk so below

In [8]:
import os

# Define the folder name where we want to save our database files
DB_DIR = "faiss_extract_once_db"

print(f"\nSerializing FAISS index to disk at ./{DB_DIR} ...")

# Save the index. This creates the .faiss and .pkl files.
vectorstore.save_local(DB_DIR)

# Verify the files were created
print("Listing files in the directory:")
print(os.listdir(DB_DIR))


Serializing FAISS index to disk at ./faiss_extract_once_db ...
Listing files in the directory:
['index.faiss', 'index.pkl']


#delete from ram now since we stored locally

In [9]:
print("\nSimulating Server Restart...")
# Delete the in-memory variable
del vectorstore

print("Loading FAISS index directly from hard drive...\n")

# We must explicitly allow dangerous deserialization.
# Pickle files can execute malicious code if you download them from an untrusted source.
# Because WE created this index locally, it is 100% safe to load.
restored_db = FAISS.load_local(
    folder_path=DB_DIR,
    embeddings=embedder,
    allow_dangerous_deserialization=True 
)

# Test the restored database
restored_results = restored_db.similarity_search("Why are documents failing?", k=1)



for i ,doc in enumerate(restored_results):
    print(doc.page_content)
    print(doc.metadata)


print("-------------\n")


print("=== RESTORED DATABASE SEARCH ===")
print(f"Content: {restored_results[0].page_content}")
print(f"Metadata: {restored_results[0].metadata}")
print("\n[System] Pipeline complete. The index is fully persistent.")


Simulating Server Restart...
Loading FAISS index directly from hard drive...

Marten noted a critical issue: 15% of scanned PDF documents are failing the OCR pipeline.
{'source': 'Extract_Once_Audit.pdf', 'chapter': 'Compliance', 'chunk_id': 'CHK-002'}
-------------

=== RESTORED DATABASE SEARCH ===
Content: Marten noted a critical issue: 15% of scanned PDF documents are failing the OCR pipeline.
Metadata: {'source': 'Extract_Once_Audit.pdf', 'chapter': 'Compliance', 'chunk_id': 'CHK-002'}

[System] Pipeline complete. The index is fully persistent.
